# 03_text_representation: TF-IDF, Cosine Similarity, Hashing Vectorizer, and BM25 from Scratch

This notebook implements the math behind text representation models. It builds a manual TF-IDF vectorizer, performs L2 normalization, computes cosine similarity, implements Okapi BM25 scoring with length penalty parameters, and simulates the Feature Hashing trick with sign hashing to balance collisions.

In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# Define tiny corpus matching study guide
corpus = ["cat feline", "feline rug"]

# 1. Scikit-learn TF-IDF Vectorizer matching our math formulation
vectorizer = TfidfVectorizer(norm='l2', smooth_idf=True, sublinear_tf=False)
tfidf_matrix = vectorizer.fit_transform(corpus).toarray()

print("Vocabulary order:\n", vectorizer.vocabulary_)
print("\nTF-IDF Vectors:")
for doc, vec in zip(corpus, tfidf_matrix):
    print(f"  '{doc}' -> {np.round(vec, 4)}")

# Calculate Cosine Similarity
cos_sim = np.dot(tfidf_matrix[0], tfidf_matrix[1])
print(f"\nComputed Cosine Similarity: {cos_sim:.4f}")

# Assertions checking matching math outputs to 4 decimal places
np.testing.assert_almost_equal(tfidf_matrix[0, 0], 0.8148, decimal=4) # cat
np.testing.assert_almost_equal(tfidf_matrix[0, 1], 0.5797, decimal=4) # feline
np.testing.assert_almost_equal(cos_sim, 0.3361, decimal=4)

Vocabulary order:
 {'cat': 0, 'feline': 1, 'rug': 2}

TF-IDF Vectors:
  'cat feline' -> [0.8148 0.5797 0.    ]
  'feline rug' -> [0.     0.5797 0.8148]

Computed Cosine Similarity: 0.3361


### Output Explanation: TF-IDF and Cosine Similarity
- **L2 Normalization:** Each raw TF-IDF vector is scaled by its Euclidean length, mapping the vectors to a unit hypersphere. The normalized coordinates for Document 1 are `[0.8148, 0.5797, 0.0]`, which matches our hand calculation.
- **Similarity Metric:** Since the vectors are pre-normalized, the cosine similarity simplifies to the dot product, yielding `0.3361` due to the shared token `"feline"`.

In [2]:
def compute_bm25_score(tf, doc_len, avgdl, idf, k1=1.2, b=0.75):
    numerator = tf * (k1 + 1)
    denominator = tf + k1 * (1.0 - b + b * (doc_len / avgdl))
    return idf * (numerator / denominator)

# Setup inputs matching study guide
doc1_len = 2 # "cat feline"
doc2_len = 4 # "feline rug garden cat"
avgdl = 3.0
idf_cat = 1.40
k1, b = 1.2, 0.75

# Document term frequencies for query Q = {"cat"}
tf_doc1 = 1
tf_doc2 = 1

score_d1 = compute_bm25_score(tf_doc1, doc1_len, avgdl, idf_cat, k1, b)
score_d2 = compute_bm25_score(tf_doc2, doc2_len, avgdl, idf_cat, k1, b)

print(f"BM25 Score for Document 1 (cat feline): {score_d1:.4f}")
print(f"BM25 Score for Document 2 (feline rug garden cat): {score_d2:.4f}")

# Assertions verifying length normalization impact
np.testing.assert_almost_equal(score_d1, 1.6211, decimal=4)
np.testing.assert_almost_equal(score_d2, 1.2320, decimal=4)
assert score_d1 > score_d2, "Short document should score higher for identical word frequencies!"

BM25 Score for Document 1 (cat feline): 1.6211
BM25 Score for Document 2 (feline rug garden cat): 1.2320


### Output Explanation: BM25 Scoring
- **Length Penalty Impact:** Document 1 ($D_1$) scores higher than Document 2 ($D_2$) even though both contain the query token `"cat"` exactly once. The length normalization parameter $b = 0.75$ scales down the score of the longer document ($D_2$) because its words are diluted.

In [3]:
import hashlib

def hash_word(word, B):
    # Compute index bucket using md5
    h = int(hashlib.md5(word.encode('utf-8')).hexdigest(), 16)
    idx = h % B
    # Compute sign hash (+1 or -1)
    sign = 1 if (h // B) % 2 == 0 else -1
    return idx, sign

B = 1000  # Number of buckets
words = ["purchase", "buy", "cat", "purchase"]

hash_vector = np.zeros(B)
for w in words:
    idx, sign = hash_word(w, B)
    hash_vector[idx] += sign
    print(f"Word: '{w:<8}' -> Bucket: {idx:<3} | Sign: {sign:+2}")

print(f"\nNon-zero indices in hash vector: {np.where(hash_vector != 0)[0]}")
assert hash_vector.sum() != 0, "Feature Hashing vector is empty!"

Word: 'purchase' -> Bucket: 589 | Sign: +1
Word: 'buy     ' -> Bucket: 661 | Sign: +1
Word: 'cat     ' -> Bucket: 632 | Sign: +1
Word: 'purchase' -> Bucket: 589 | Sign: +1

Non-zero indices in hash vector: [589 632 661]


### Output Explanation: Feature Hashing
- **Bucket Mapping:** Words are mapped to a fixed vector space of size $B = 1000$ using `hashlib.md5`. This bypasses the need to store a dictionary table in memory.
- **Sign Hash:** Collisions cancel out on average because words are randomly scaled by $+1$ or $-1$ before addition, ensuring expected representation bias remains near 0.